In [21]:
import osmnx as ox
import matplotlib.pyplot as plt
import seaborn as sns
import h3
import pandas as pd
import numpy as np
import os
from pathlib import Path
import geopandas as gpd
import contextily as ctx
import io, zipfile, requests
import shapely

import folium
import mapclassify

# Create hourly aggregated Dateframe
Create a new df with hourly timestamps in the range of the dataset and all unique H3 indices so we have a complete grid for analysis

In [22]:
# TODO: Validate data set paths
taxi_data_ca = pd.read_parquet("../data/processed/taxi_data_processed_big.parquet")
taxi_data_census = pd.read_parquet('../data/processed/taxi_data_processed_small.parquet')
#poi_data_processed = pd.read_csv('../data/processed/chicago_pois_agg.csv')
# weather_data_processed = pd.read_csv('../data/processed/weather_data_processed.csv')

## Spatial Chicago Ground Truth
Build a combined lookup with h3_8 as the finest level, matched to h3_7, census tract (geoid10) and community area (commarea).

In [23]:
# Chicago census tract boundaries (2010) — download from the city data portal and cache locally
C_PATH = Path('..') / 'data' / 'chicago_census_comm.gpkg'

if C_PATH.exists():
    census_tracts = gpd.read_file(C_PATH)
else:
    print('Downloading Chicago census tract and community area boundaries...')
    url = 'https://data.cityofchicago.org/resource/74p9-q2aq.geojson?$limit=1000'
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    census_tracts = gpd.read_file(io.BytesIO(r.content))
    census_tracts.to_file(C_PATH, driver='GPKG')

    print(f'Saved {len(census_tracts)} census tracts to {C_PATH}')

all_census_comm = census_tracts.to_crs(epsg=4326)
print(f'Census tracts loaded: {len(all_census_comm)}')

Census tracts loaded: 801


In [24]:
# Dissolve the community areas into a single boundary for the whole city
chicago_boundary = all_census_comm.union_all()

# Finest level: all H3 res-8 cells covering Chicago (h3 v4 API replaces v3 polyfill)
h3_index_8 = sorted(h3.geo_to_cells(chicago_boundary, 8))
all_census_tracts = pd.DataFrame({'h3_index_8': h3_index_8})

# h3_7 is simply the res-7 parent of each res-8 cell
all_census_tracts['h3_index_7'] = all_census_tracts['h3_index_8'].apply(
    lambda c: h3.cell_to_parent(c, 7)
)

# Match census tract + community area via the location of each hex centroid
centroids = all_census_tracts['h3_index_8'].apply(h3.cell_to_latlng)
hex_points = gpd.GeoDataFrame(
    all_census_tracts,
    geometry=gpd.points_from_xy(
        centroids.apply(lambda x: x[1]),  # lng
        centroids.apply(lambda x: x[0]),  # lat
    ),
    crs='EPSG:4326',
)
# Perform a spatial join to match each hexagon centroid to the corresponding census tract and community area
all_census_tracts = gpd.sjoin(
    hex_points,
    all_census_comm[['geoid10', 'commarea', 'geometry']],
    how='left',
    predicate='within',
).drop(columns=['index_right', 'geometry'])

# One row per h3_8 cell, everything matched against it
all_census_tracts = pd.DataFrame(all_census_tracts).reset_index(drop=True)

print(
    f"{len(all_census_tracts)} h3_8 cells | "
    f"{all_census_tracts['h3_index_7'].nunique()} h3_7 parents | "
    f"{all_census_tracts['geoid10'].nunique()} census tracts | "
    f"{all_census_tracts['commarea'].nunique()} community areas"
)
all_census_tracts.head()

849 h3_8 cells | 153 h3_7 parents | 534 census tracts | 77 community areas


,h3_index_8,h3_index_7,geoid10,commarea
0,8826641901fffff,872664190ffffff,17031838800,51
1,8826641903fffff,872664190ffffff,17031838800,51
2,8826641905fffff,872664190ffffff,17031838800,51
3,8826641907fffff,872664190ffffff,17031838800,51
4,8826641909fffff,872664190ffffff,17031838800,51


## Temporal Ground truth

In [25]:
# Extract hour from Trip Start Timestamp and Trip End Timestamp for temporal analysis
taxi_data_census['Pickup Hour'] = pd.to_datetime(taxi_data_census['Trip Start Timestamp']).dt.floor('h')
taxi_data_census['Dropoff Hour'] = pd.to_datetime(taxi_data_census['Trip End Timestamp']).dt.floor('h')

# Get the range of timestamps in the dataset
min_timestamp = taxi_data_census['Pickup Hour'].min()
max_timestamp = taxi_data_census['Dropoff Hour'].max()

# Create a complete hourly timestamp range and a daily timestamp range for the entire dataset
hourly_timestamps = pd.date_range(start=min_timestamp, end=max_timestamp, freq='h')
daily_timestamps = pd.date_range(start=min_timestamp, end=max_timestamp, freq='D')


/var/folders/mb/d7qxmv150dz1wszhty3l250c0000gn/T/ipykernel_75130/918921653.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  taxi_data_census['Pickup Hour'] = pd.to_datetime(taxi_data_census['Trip Start Timestamp']).dt.floor('h')
/var/folders/mb/d7qxmv150dz1wszhty3l250c0000gn/T/ipykernel_75130/918921653.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  taxi_data_census['Dropoff Hour'] = pd.to_datetime(taxi_data_census['Trip End Timestamp']).dt.floor('h')


## Spatial Temaporal Discretization
Now we use the Spatial Ground Truth to aggregate hour data on different spatial and temporal granuality levels.

In [26]:
# Define aggregations specifically for the Pickups
pickup_aggregations = {
    'Total_Trip_Start': ('Trip ID', 'count'),
    'Unique Taxis': ('Taxi ID', 'nunique'),
    'AvgTripSeconds': ('Trip Seconds', 'mean'),
    'AvgTripMiles': ('Trip Miles', 'mean'),
    'AvgFare': ('Trip Total', 'mean'),
    'MostCommonCompany': ('Company', lambda x: x.mode()[0] if not x.mode().empty else np.nan),
    'CompanyCount': ('Company', 'nunique'),
    # TODO: Mean might not be the best way to aggregate lat/lon values, consider using the centroid of the H3 cell instead
    'PickupLongitude': ('Pickup Centroid Longitude', 'mean'),
    'PickupLatitude': ('Pickup Centroid Latitude', 'mean'), 
}

# Define aggregations specifically for the Dropoffs
dropoff_aggregations = {
    'Total_Trip_End': ('Trip ID', 'count')
}

In [27]:
# Map each spatial level to its pickup column, dropoff column, and the grid/universe column.
# Pickups and dropoffs live in SEPARATE columns, so each side is grouped on its own key.
spatial_map = {
    'h3_8':      {'pickup': 'h3_index_pickup_8',    'dropoff': 'h3_index_dropoff_8',    'grid': 'h3_index_8'},
    'h3_7':      {'pickup': 'h3_index_pickup_7',    'dropoff': 'h3_index_dropoff_7',    'grid': 'h3_index_7'},
    'census':    {'pickup': 'pickup_tract',         'dropoff': 'dropoff_tract',         'grid': 'geoid10'},
    'community': {'pickup': 'pickup_comm',          'dropoff': 'dropoff_comm',          'grid': 'commarea'},
}

# The census/community codes are float64 in the taxi data (e.g. 8.0) but strings in the
# gpkg-derived grid (e.g. '8'). Normalize them to matching string codes so the merge hits.
# Derived from the original float columns, so this stays correct if the cell is re-run.
def _to_code(s):
    return s.astype('Int64').astype('str')

taxi_data_census['pickup_tract']  = _to_code(taxi_data_census['Pickup Census Tract'])
taxi_data_census['dropoff_tract'] = _to_code(taxi_data_census['Dropoff Census Tract'])
taxi_data_census['pickup_comm']   = _to_code(taxi_data_census['Pickup Community Area'])
taxi_data_census['dropoff_comm']  = _to_code(taxi_data_census['Dropoff Community Area'])

# Temporal levels: the aggregation key must be floored to the SAME granularity as the grid,
# otherwise a daily grid merged against hour-floored keys only matches the midnight hour.
taxi_data_census['Pickup Day']  = taxi_data_census['Pickup Hour'].dt.floor('D')
taxi_data_census['Dropoff Day'] = taxi_data_census['Dropoff Hour'].dt.floor('D')
temporal_map = {
    'hourly': {'range': hourly_timestamps, 'pickup': 'Pickup Hour', 'dropoff': 'Dropoff Hour'},
    'daily':  {'range': daily_timestamps,  'pickup': 'Pickup Day',  'dropoff': 'Dropoff Day'},
}

count_columns = ['Total_Trip_Start', 'Unique Taxis', 'CompanyCount', 'Total_Trip_End']

for spatial_level, s in spatial_map.items():
    for temporal_level, t in temporal_map.items():
        print(f"Processing {spatial_level} spatial aggregation with {temporal_level} temporal aggregation...")
        grid_col = s['grid']

        # Complete spatial universe of Chicago from the master lookup (covers every cell/tract/area,
        # including ones with no trips in the current split).
        unique_spatial = all_census_tracts[grid_col].dropna().drop_duplicates()
        complete_grid = pd.MultiIndex.from_product(
            [t['range'], unique_spatial], names=['timestamp', grid_col]
        ).to_frame(index=False)

        # Aggregate pickups on the pickup key, dropoffs on the dropoff key
        aggregated_pickups = taxi_data_census.groupby(
            [t['pickup'], s['pickup']]
        ).agg(**pickup_aggregations).reset_index()
        aggregated_dropoffs = taxi_data_census.groupby(
            [t['dropoff'], s['dropoff']]
        ).agg(**dropoff_aggregations).reset_index()

        # Merge pickups: drop only the redundant time column, KEEP the spatial key for the next merge
        complete_grid = complete_grid.merge(
            aggregated_pickups,
            left_on=['timestamp', grid_col],
            right_on=[t['pickup'], s['pickup']],
            how='left',
        ).drop(columns=[t['pickup']])
        if s['pickup'] != grid_col:
            complete_grid = complete_grid.drop(columns=[s['pickup']])

        # Merge dropoffs the same way
        complete_grid = complete_grid.merge(
            aggregated_dropoffs,
            left_on=['timestamp', grid_col],
            right_on=[t['dropoff'], s['dropoff']],
            how='left',
        ).drop(columns=[t['dropoff']])
        if s['dropoff'] != grid_col:
            complete_grid = complete_grid.drop(columns=[s['dropoff']])

        # Counts of empty cells are true zeros; the mode-company gets an empty string;
        # averages are left as NaN (no trips -> no meaningful average).
        complete_grid[count_columns] = complete_grid[count_columns].fillna(0)
        complete_grid['MostCommonCompany'] = complete_grid['MostCommonCompany'].fillna("")

        complete_grid.to_parquet(
            f'../data/processed/complete_grid_{spatial_level}_{temporal_level}.parquet', index=False)
        print(f"Completed processing for {spatial_level} with {temporal_level}. ({len(complete_grid)} rows)")

print("All spatial and temporal aggregation levels processed successfully.")

Processing h3_8 spatial aggregation with hourly temporal aggregation...
Completed processing for h3_8 with hourly. (17972481 rows)
Processing h3_8 spatial aggregation with daily temporal aggregation...
Completed processing for h3_8 with daily. (749667 rows)
Processing h3_7 spatial aggregation with hourly temporal aggregation...
Completed processing for h3_7 with hourly. (3238857 rows)
Processing h3_7 spatial aggregation with daily temporal aggregation...
Completed processing for h3_7 with daily. (135099 rows)
Processing census spatial aggregation with hourly temporal aggregation...
Completed processing for census with hourly. (11304246 rows)
Processing census spatial aggregation with daily temporal aggregation...
Completed processing for census with daily. (471522 rows)
Processing community spatial aggregation with hourly temporal aggregation...
Completed processing for community with hourly. (1630013 rows)
Processing community spatial aggregation with daily temporal aggregation...
Com